In [8]:
import pandas as pd

# 1) Read the dataset
df = pd.read_csv("final_merged_data.csv")

# 2) Process the time column: convert to datetime and extract hour and day_of_week
df["last_reported"] = pd.to_datetime(df["last_reported"], errors="coerce")
df["hour"] = df["last_reported"].dt.hour
df["day_of_week"] = df["last_reported"].dt.dayofweek

# 3) Define the target column for the regression task
target_col = "num_bikes_available"

# 4) Select only the specified features and the target column
feature_cols = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "max_air_temperature_celsius",
    "max_relative_humidity_percent",
    "max_barometric_pressure_hpa",
]
keep_cols = feature_cols + [target_col]

df_selected = df[keep_cols].copy()

# 5) Remove rows with missing values in the selected columns
df_clean = df_selected.dropna()

# 6) Print an overview of the cleaned data
print("Cleaned data shape:", df_clean.shape)
print("\nFirst 5 rows:")
print(df_clean.head())

Cleaned data shape: (298946, 8)

First 5 rows:
   station_id  hour  day_of_week  month  max_air_temperature_celsius  \
0          10     0            6     12                        14.01   
1         100     0            6     12                        14.01   
2         109     0            6     12                        14.01   
3          11     0            6     12                        14.01   
4         114     0            6     12                        14.01   

   max_relative_humidity_percent  max_barometric_pressure_hpa  \
0                           84.3                      1002.56   
1                           84.3                      1002.56   
2                           84.3                      1002.56   
3                           84.3                      1002.56   
4                           84.3                      1002.56   

   num_bikes_available  
0                   15  
1                   17  
2                   20  
3                    1  
4   

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1) Split the features and target
X = df_clean[feature_cols]
y = df_clean[target_col]

# 2) Split the dataset into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3) Initialize the models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(
        max_depth=25,          
        min_samples_split=20, 
        min_samples_leaf=10,  
        random_state=42
    ),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=50,
        max_depth=25,
        min_samples_split=20,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42,
    ),
    "KNeighbors Regressor": KNeighborsRegressor(),
}

# 4) Train and evaluate each model
results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2,
    })

# 5) Summarize and display the results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="MAE", ascending=True).reset_index(drop=True)
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["R2 Score"] = results_df["R2 Score"].round(4)

print("Model Evaluation on Test Set (sorted by MAE):")
display(results_df)

# 6) Print Random Forest metrics for quick regression quality check
rf_row = results_df.loc[results_df["Model"] == "Random Forest Regressor"].iloc[0]
print("\nCompressed Random Forest Metrics on Test Set:")
print(f"R2 Score: {rf_row['R2 Score']}")
print(f"MAE: {rf_row['MAE']}")
print(f"RMSE: {rf_row['RMSE']}")

Model Evaluation on Test Set (sorted by MAE):


,Model,MAE,RMSE,R2 Score
0,Random Forest Regressor,1.9903,3.0434,0.9021
1,Decision Tree Regressor,2.2121,3.8201,0.8458
2,KNeighbors Regressor,4.0336,5.8257,0.6414
3,Linear Regression,8.1363,9.7279,-0.0000



Compressed Random Forest Metrics on Test Set:
R2 Score: 0.9021
MAE: 1.9903
RMSE: 3.0434


In [10]:
import joblib
from pathlib import Path

# [核心逻辑修改]：不再强行指定 RF，而是自动获取得分最高（MAE最小）的模型名
best_model_name = results_df.iloc[0]["Model"]
print(f"\n The Best Model is: {best_model_name}")

best_model = models[best_model_name]

# Make save path robust to notebook working directory
cwd = Path.cwd()
if cwd.name == "ml":
    output_path = cwd / "best_bike_model.joblib"
else:
    output_path = cwd / "ml" / "best_bike_model.joblib"

output_path.parent.mkdir(parents=True, exist_ok=True)

# 动态保存冠军模型
joblib.dump(best_model, output_path, compress=3)

print(f"Compressed {best_model_name} model has been saved to {output_path}")


 The Best Model is: Random Forest Regressor
Compressed Random Forest Regressor model has been saved to /Users/flacko/Documents/ComputerScience/Project/comp30830/dublin-bikes-webapp/ml/best_bike_model.joblib
